# Karpathy's AutoResearch on Google Colab

Autonomous AI agents experiment with LLM training. Each run modifies `train.py`, trains for 5 minutes, evaluates, and iterates.

### Setup
1. Go to **Runtime > Change runtime type > GPU**
2. Run the cells below in order

In [ ]:
# Verify GPU is available
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Mount Google Drive to persist data/results between sessions
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo
!git clone https://github.com/karpathy/autoresearch.git /content/autoresearch
%cd /content/autoresearch

In [ ]:
# Install uv package manager
!curl -LsSf https://astral.sh/uv/install.sh | sh

import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

In [ ]:
# Install project dependencies
!uv sync

In [ ]:
# Symlink cache to Google Drive so data persists between sessions
import os

drive_cache = '/content/drive/MyDrive/autoresearch_cache'
local_cache = os.path.expanduser('~/.cache/autoresearch')

os.makedirs(drive_cache, exist_ok=True)

if os.path.islink(local_cache):
    print(f'Symlink already exists: {local_cache} -> {os.readlink(local_cache)}')
elif os.path.exists(local_cache):
    print(f'Warning: {local_cache} already exists as a real directory. Skipping symlink.')
else:
    os.makedirs(os.path.dirname(local_cache), exist_ok=True)
    os.symlink(drive_cache, local_cache)
    print(f'Created symlink: {local_cache} -> {drive_cache}')

In [ ]:
# Prepare data: downloads shards + trains tokenizer (~2 minutes first time)
!uv run prepare.py

In [ ]:
# Fix batch size for T4/A100-40GB (original 128 causes OOM)
!sed -i 's/DEVICE_BATCH_SIZE = 128/DEVICE_BATCH_SIZE = 32/' /content/autoresearch/train.py

In [ ]:
# Run baseline training (~5 minutes + startup/eval overhead)
!uv run train.py

---
## Experiment Loop Setup

After the baseline runs successfully, run the cells below to set up the experiment infrastructure.

In [ ]:
# Set up git and create experiment branch
!git config user.email "autoresearch@colab"
!git config user.name "autoresearch"

from datetime import datetime
tag = datetime.now().strftime('%b%d').lower()
branch = f'autoresearch/{tag}'
print(f'Creating branch: {branch}')
!git checkout -b {branch}

In [ ]:
# Initialize results.tsv and record baseline
# Update the val_bpb and memory_gb values from your baseline run above

BASELINE_VAL_BPB = "1.089057"    # <-- update from your baseline output
BASELINE_MEMORY_GB = "11.4"       # <-- update (peak_vram_mb / 1024)

with open('/content/autoresearch/results.tsv', 'w') as f:
    f.write('commit\tval_bpb\tmemory_gb\tstatus\tdescription\n')
    commit = !git rev-parse --short HEAD
    f.write(f'{commit[0]}\t{BASELINE_VAL_BPB}\t{BASELINE_MEMORY_GB}\tkeep\tbaseline (batch_size=32)\n')

!cat /content/autoresearch/results.tsv

In [ ]:
# Helper function: run an experiment, log results, keep or revert
import subprocess, re, os

def run_experiment(description):
    """Commit changes, run training, log results, keep or revert."""
    # Commit the current changes
    subprocess.run(['git', 'add', 'train.py'], cwd='/content/autoresearch')
    subprocess.run(['git', 'commit', '-m', description], cwd='/content/autoresearch')
    commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                           capture_output=True, text=True, cwd='/content/autoresearch').stdout.strip()

    # Run training
    print(f'\n=== Running experiment: {description} ===')
    print(f'Commit: {commit}')
    print('Training for ~5 minutes...\n')

    result = subprocess.run(['uv', 'run', 'train.py'],
                           capture_output=True, text=True, cwd='/content/autoresearch',
                           timeout=600)
    output = result.stdout + result.stderr

    # Parse results
    val_match = re.search(r'val_bpb:\s+(\S+)', output)
    mem_match = re.search(r'peak_vram_mb:\s+(\S+)', output)

    if val_match and mem_match:
        val_bpb = float(val_match.group(1))
        memory_gb = round(float(mem_match.group(1)) / 1024, 1)
        print(f'\n=== Results ===')
        print(f'val_bpb:    {val_bpb:.6f}')
        print(f'memory_gb:  {memory_gb}')

        # Read best so far from results.tsv
        best_bpb = float('inf')
        with open('/content/autoresearch/results.tsv') as f:
            for line in f.readlines()[1:]:
                parts = line.strip().split('\t')
                if parts[3] == 'keep':
                    best_bpb = min(best_bpb, float(parts[1]))

        if val_bpb < best_bpb:
            status = 'keep'
            print(f'IMPROVED! {best_bpb:.6f} -> {val_bpb:.6f} (keeping)')
        else:
            status = 'discard'
            print(f'No improvement ({val_bpb:.6f} >= {best_bpb:.6f}), reverting.')
            subprocess.run(['git', 'reset', '--hard', 'HEAD~1'], cwd='/content/autoresearch')
    else:
        # Crash
        val_bpb = 0.0
        memory_gb = 0.0
        status = 'crash'
        print(f'\n=== CRASH ===')
        # Print last 30 lines of output for debugging
        print('\n'.join(output.strip().split('\n')[-30:]))
        subprocess.run(['git', 'reset', '--hard', 'HEAD~1'], cwd='/content/autoresearch')

    # Log to results.tsv
    with open('/content/autoresearch/results.tsv', 'a') as f:
        f.write(f'{commit}\t{val_bpb:.6f}\t{memory_gb}\t{status}\t{description}\n')

    return {'val_bpb': val_bpb, 'memory_gb': memory_gb, 'status': status}

print('run_experiment() ready. Use it after editing train.py.')

---
## Run Experiments

For each experiment:
1. Edit `train.py` in the cell below (modify hyperparams, architecture, etc.)
2. Run the experiment cell
3. Check results — improvements are kept, failures are reverted

### Experiment 1: Increase depth from 8 to 12

In [ ]:
# Edit train.py — change what you want to experiment with
# This example increases model depth from 8 to 12 layers
!sed -i 's/^DEPTH = 8/DEPTH = 12/' /content/autoresearch/train.py

# Verify the change
!grep -n 'DEPTH\|DEVICE_BATCH_SIZE\|MATRIX_LR' /content/autoresearch/train.py | head -5

In [ ]:
# Run the experiment (~5 min)
run_experiment('increase depth from 8 to 12')

### Experiment 2: Your next idea
Duplicate the two cells above and try something new:
- Change learning rates
- Adjust model width (ASPECT_RATIO)
- Try different WINDOW_PATTERN
- Modify WEIGHT_DECAY, ADAM_BETAS
- Change warmup/warmdown ratios

In [ ]:
# View current results
import pandas as pd
df = pd.read_csv('/content/autoresearch/results.tsv', sep='\t')
print(df.to_string(index=False))
print(f"\nBest val_bpb: {df[df['status']=='keep']['val_bpb'].min():.6f}")

In [ ]:
# Save results to Google Drive (in case Colab disconnects)
!cp /content/autoresearch/results.tsv /content/drive/MyDrive/autoresearch_cache/results.tsv
print('Results saved to Google Drive')